# SFT-разогрев T-lite с LoRA

Решение 03.11.2026 (`docs/decisions.md`, раздел 9). Примеры собираются в токены функцией среды, как роллауты RL;
loss — только по репликам модели; `transformers.Trainer` + LoRA (peft), логиты — только в позициях с меткой.
Отложено 9 тем SFT (с ними — задания тех же семейств): по loss на них отмечается лучшая эпоха. Результат —
`runs/sft/<дата>/` (адаптер, журнал, разбиение; при `save_epochs` — адаптеры всех эпох в `epochs/`).

Перед запуском: `pip install --user -e ".[models,training]"`; веса T-lite должны быть доступны (Hugging Face или
локальный кеш в `HF_HUB_CACHE`).

In [1]:
import json, os, sys, time
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # перед запуском проверить nvidia-smi
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
import torch
from presentation_rl.environment import build_prompt
from presentation_rl.model_loader import load_model
from presentation_rl.prompts import system_prompt
from presentation_rl.training import sft

CONFIG = "sft_tlite_v2.yaml"  # EXP-020; прежние запуски — sft_tlite.yaml (3 эпохи), sft_tlite_7ep.yaml
config = sft.SFTConfig.load(ROOT / "configs" / "experiments" / CONFIG)
OUT = ROOT / "runs" / "sft" / time.strftime("%Y%m%d_%H%M")
OUT.mkdir(parents=True, exist_ok=True)
print(config, OUT, torch.cuda.get_device_name(0), sep="\n")

mkdir -p failed for path /home/jovyan/.config/matplotlib: [Errno 20] Not a directory: '/home/jovyan/.config/matplotlib'
Matplotlib created a temporary cache directory at /tmp/matplotlib-qxvry7qr because there was an issue with the default path (/home/jovyan/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


SFTConfig(model_id='t-tech/T-lite-it-2.1', revision='main', lora_rank=32, lora_alpha=32, learning_rate=0.0001, epochs=6, grad_accum=8, warmup_ratio=0.03, eval_tasks=9, seed=0, max_tokens=16000, eval_topics=['t006', 't034', 't039', 't050', 't052', 't054', 't062', 't063', 't066'], save_epochs=True)
/home/jovyan/Diplom/runs/sft/20261111_2313
Tesla V100-SXM3-32GB


In [2]:
start = time.time()
model, tokenizer = load_model(config.model_id, config.revision, dtype="float16", device_map={"": 0})
print(f"загрузка {time.time() - start:.0f} с, память {torch.cuda.memory_allocated() / 2**30:.1f} ГБ")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

загрузка 741 с, память 15.3 ГБ


## Примеры токенами

Эпизод каждого примера собирается так же, как роллаут: начало — шаблон чата (системный промпт, инструменты, запрос),
дальше реплики модели и выдача инструментов в порядке среды. Слишком длинные эпизоды не обрезаются (оборвался бы
ответ), а отбрасываются — их список печатается.

In [3]:
def prompt(request):
    return build_prompt(tokenizer, system_prompt(request), request)


samples = sft.build_samples(ROOT, tokenizer, prompt)
samples, dropped = sft.drop_long(samples, config.max_tokens)
print("отброшены как слишком длинные:", [(s.id, s.kind, len(s.ids)) for s in dropped])
train_samples, eval_samples = sft.split(samples, config.eval_tasks, config.seed, config.eval_topics,
                                         sft.task_families(ROOT))
for name, part in (("обучение", train_samples), ("отложенные", eval_samples)):
    print(name, sft.lengths(part))
(OUT / "split.json").write_text(json.dumps({"eval_topics": sorted({s.id for s in eval_samples}),
                                            "dropped": [[s.id, s.kind, len(s.ids)] for s in dropped]},
                                           ensure_ascii=False, indent=1), encoding="utf-8")

отброшены как слишком длинные: []
обучение {'examples': 132, 'max': 12085, 'median': 4587, 'total': 715158, 'target_total': 243786}
отложенные {'examples': 16, 'max': 10874, 'median': 5296, 'total': 103812, 'target_total': 33388}


129

## Обучение

In [4]:
start = time.time()
torch.cuda.reset_peak_memory_stats()
model = sft.train(config, train_samples, eval_samples, OUT, model=model, tokenizer=tokenizer)
print(f"обучение {(time.time() - start) / 60:.0f} мин, пик памяти {torch.cuda.max_memory_allocated() / 2**30:.1f} ГБ")
log = json.loads((OUT / "log.json").read_text(encoding="utf-8"))
for row in log:
    if "eval_loss" in row:
        print(f"эпоха {row['epoch']:.0f}: loss на отложенных темах {row['eval_loss']:.4f}")
print("адаптеры эпох:", sorted(p.name for p in (OUT / "epochs").glob("epoch_*")))

trainable params: 87,293,952 || all params: 8,276,005,888 || trainable%: 1.0548


[W1111 23:26:52.594565218 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1927282688 bytes (free: 464519168, total: 34072559616).


Epoch,Training Loss,Validation Loss
1,1.593615,1.499657
2,1.440521,1.452037
3,1.145691,1.426888
4,1.188477,1.421675
5,1.303145,1.419240
6,1.210218,1.419604


[W1111 23:29:56.513437841 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2157969408 bytes (free: 1699741696, total: 34072559616).
[W1111 23:33:08.209379378 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2246049792 bytes (free: 732954624, total: 34072559616).
[W1111 23:34:56.624453453 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2319450112 bytes (free: 330301440, total: 34072559616).
[W1111 23:36:51.015643229 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2323644416 bytes (free: 93323264, total: 34072559616).


обучение 86 мин, пик памяти 24.8 ГБ
эпоха 1: loss на отложенных темах 1.4997
эпоха 2: loss на отложенных темах 1.4520
эпоха 3: loss на отложенных темах 1.4269
эпоха 4: loss на отложенных темах 1.4217
эпоха 5: loss на отложенных темах 1.4192
эпоха 6: loss на отложенных темах 1.4196
адаптеры эпох: ['epoch_1', 'epoch_2', 'epoch_3', 'epoch_4', 'epoch_5', 'epoch_6']


## Быстрая проверка

Один ответ на отложенную тему без поиска (полная проверка на validation — EXP-015, отдельная тетрадка).

In [5]:
from presentation_rl.dsl import parse
from presentation_rl.environment import run_episode
from presentation_rl.model_loader import hf_generator

model.eval()
model.config.use_cache = True
record = next(json.loads(l) for l in (ROOT / sft.PLAIN).read_text(encoding="utf-8").splitlines()
              if json.loads(l)["id"] in {s.id for s in eval_samples})
from presentation_rl.tasks import load_tasks
request = sft.request_for(record, {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json", split="sft")})
episode = run_episode(prompt(request), tokenizer, hf_generator(model, tokenizer, temperature=0.9), None, max_new_tokens=6000)
deck = parse(episode.answer)
print(f"{record['id']}: слайдов {len(deck.slides)}, корректных строк {deck.parse_rate:.0%}, конец — {episode.finish}")
print(episode.answer[:3000])

t006: слайдов 4, корректных строк 10%, конец — length
#PLAN: Эпоха Древнего Востока → Древний Вавилон → Древний Иран → Древняя Индия → Древний Китай → Массовое сельское хозяйство и сложение государства → Имперская интеграция и долговое рабство → Духовная сфера и наследие → Определение проблемы исследования
#SLIDE
#TITLE: История Древнего Востока: хронология, периодизация, особенности развития
#TEXT: Рассмотрим крупные цивилизации Древнего Востока, их хронологию, периодизацию и общие культурные особенности. Основные цивилизации: Вавилон, Иран, Индия и Китай. Критерий сравнения — характер государственных институтов и их эволюция.
#DIAGRAM: type=tree; Два показателя различия государств Древнего Востока > Политическая организация, Экономическая организация; Политическая организация > Монолитная империя, Конфедерация; Экономическая организация > Массовое натуральное производство, Торовое хозяйство
#SLIDE
#TITLE: Эпоха Древнего Востока: территориальный масштаб и хронология
#BULLET: Греко-рим